In [2]:
import transformers
import datasets
import accelerate
import evaluate
import tqdm
print("Transformers version:", transformers.__version__)
print("Datasets version:", datasets.__version__)
print("Accelerate version:", accelerate.__version__)
print("Evaluate version:", evaluate.__version__)

d:\conda\envs\huggingface\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Transformers version: 4.51.3
Datasets version: 3.5.0
Accelerate version: 1.5.2
Evaluate version: 0.4.3


In [3]:
from datasets import load_dataset

# 加载 WMT19 英语-法语数据集
dataset = load_dataset("wmt19", "zh-en")

print(dataset)
#分割
train_val_split=dataset["train"].train_test_split(test_size=0.01,seed=42)
small_train_dataset=train_val_split["test"]
test_dataset=dataset["validation"]
print(small_train_dataset)
print(test_dataset)

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 25984574
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 3981
    })
})
Dataset({
    features: ['translation'],
    num_rows: 259846
})
Dataset({
    features: ['translation'],
    num_rows: 3981
})


In [4]:
import os
source_lang="zh"
target_lang="en"
#从训练集中提取源语言序列

zh_tokenizer_file='zh_tokenizer.txt'
en_tokenizer_file='en_tokenizer.txt'
source_sentences=[item['translation'][source_lang] for item in small_train_dataset]
#从训练集中提取目标语言序列
target_sentences=[item['translation'][target_lang] for item in small_train_dataset]

num_samples=100000
source_sentences=source_sentences[:num_samples]
target_sentences=target_sentences[:num_samples]

with open(zh_tokenizer_file,"w",encoding="utf-8") as f:
    for sentence in source_sentences:
        f.write(sentence+"\n")
with open(en_tokenizer_file,"w",encoding="utf-8") as f:
    for sentence in target_sentences:
        f.write(sentence+'\n')

In [5]:
from tokenizers import ByteLevelBPETokenizer
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer

In [6]:
vocab_size=50000
min_frequency=2
unk_token="<unk>"
sep_token="<sep>"
pad_token="<pad>"
cls_token="<cls>"
mask_token="<mask>"
begin_seq_token='<bos>'
end_seq_token='<eos>'
special_tokens=[unk_token,sep_token,pad_token,cls_token,mask_token,begin_seq_token,end_seq_token]
#seq_token Separator Token 分隔符 分开不同的句子或者段落
#cls_token class Token 分类符

In [42]:
zh_BPE_dir='zh_BPE_tokenizer'
en_BPE_dir='en_BPE_tokenizer'
os.makedirs(zh_BPE_dir,exist_ok=True)
os.makedirs(en_BPE_dir,exist_ok=True)
#创建中文分词器
zh_tokenizer=Tokenizer(BPE())
zh_tokenizer.pre_tokenizer=Whitespace()#先按空格分割，后续交给BPE处理
trainer=BpeTrainer(
    vocab_size=50000,
    min_frequency=2,
    special_tokens=special_tokens
)
zh_tokenizer.train([zh_tokenizer_file],trainer)#训练
zh_bpe_dir='./zh_bpe'
zh_tokenizer.save(zh_bpe_dir)#保存分词器
from transformers import PreTrainedTokenizerFast
warp_zh_tokenizer=PreTrainedTokenizerFast(tokenizer_object=zh_tokenizer)
save_path='./zh_BPE_tokenizer'
warp_zh_tokenizer.save_pretrained(save_path)

('./zh_BPE_tokenizer\\tokenizer_config.json',
 './zh_BPE_tokenizer\\special_tokens_map.json',
 './zh_BPE_tokenizer\\tokenizer.json')

In [44]:
en_tokenizer=Tokenizer(BPE())
en_tokenizer.pre_tokenizer=Whitespace()
trainer=BpeTrainer(
    vocab_size=50000,
    min_frequency=2,
    special_tokens=special_tokens
)
en_tokenizer.train([en_tokenizer_file],trainer)#训练
#两种方式保存
en_bpe_dir='./en_bpe'
en_tokenizer.save(en_bpe_dir)#保存分词器
from transformers import PreTrainedTokenizerFast
warp_zh_tokenizer=PreTrainedTokenizerFast(tokenizer_object=en_tokenizer)
en_save_path='./en_BPE_tokenizer'
warp_zh_tokenizer.save_pretrained(en_save_path)

('./en_BPE_tokenizer\\tokenizer_config.json',
 './en_BPE_tokenizer\\special_tokens_map.json',
 './en_BPE_tokenizer\\tokenizer.json')

In [ ]:
# os.remove(zh_tokenizer_file)
# os.remove(en_tokenizer_file)

In [46]:
from transformers import BertTokenizer

zh_bpe_dir='./zh_BPE_tokenizer/tokenizer.json'
en_bpe_dir='./en_BPE_tokenizer/tokenizer.json'
#加载分词器
zh_load_tokenizer=BertTokenizer.from_pretrained(zh_bpe_dir)
en_load_tokenizer=BertTokenizer.from_pretrained(en_bpe_dir)

zh_encoded=zh_load_tokenizer("我爱你")
# zh_encoded=zh_load_tokenizer.encode("我爱你")

print(zh_encoded)

d:\conda\envs\huggingface\lib\site-packages\transformers\tokenization_utils_base.py:1962: FutureWarning: Calling BertTokenizer.from_pretrained() with the path to a single file or url is deprecated and won't be possible anymore in v5. Use a model identifier or the path to a directory instead.
  warnings.warn(


{'input_ids': [67435, 67432, 67432, 67432, 67433], 'token_type_ids': [0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1]}


In [ ]:
#input_ids 文本的token id
#token_type_ids 句子类型id 第一个句子为0，第二个句子为1
#attention_mask 掩码，为1时为数据，为0时为padding

In [48]:
print(f"中文分词器的'<unk>'字符：{zh_load_tokenizer.unk_token}")
print(f"英文分词器的'<unk>'字符：{en_load_tokenizer.unk_token}")
print(f"中文分词器的词汇表大小：{zh_load_tokenizer.vocab_size}")
print(f"英文分词器的词汇表大小：{en_load_tokenizer.vocab_size}")

中文分词器的'<unk>'字符：[UNK]
英文分词器的'<unk>'字符：[UNK]
中文分词器的词汇表大小：67432
英文分词器的词汇表大小：65592
